In [ ]:
# 🌌 HoroConsultant - Production Cloud Fine-Tuning Pipeline
import os
import sys
import subprocess

# Suppress PyDev / frozen modules debugger warnings on Kaggle
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ['PYTHONWARNINGS'] = 'ignore'

# 1. Load Secrets safely from Kaggle Secrets (individual try-except per key)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for secret_key in ['HF_TOKEN', 'APP_SUPABASE_URL', 'APP_SUPABASE_KEY', 'GH_TOKEN']:
        try:
            val = user_secrets.get_secret(secret_key)
            if val:
                os.environ[secret_key] = val
                print(f'✅ Kaggle Secret loaded: {secret_key}')
        except Exception as e:
            print(f'ℹ️ Kaggle Secret note ({secret_key}): {e}')
except Exception as e:
    print(f'ℹ️ Kaggle Secrets Client not available: {e}')

# 2. Safe Git Clone / Pull with pure Python subprocess
target_dir = '/kaggle/working/HoroConsultant'
if not os.path.exists(target_dir):
    print('📦 Cloning HoroConsultant repository...')
    subprocess.run(['git', 'clone', 'https://github.com/pphothidaen/HoroConsultant.git', target_dir], check=True)
else:
    print('🔄 Pulling latest updates...')
    subprocess.run(['git', '-C', target_dir, 'pull'], check=True)

os.chdir(target_dir)
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)

# 3. Install Fine-Tuning Dependencies safely without overwriting pre-installed Kaggle CUDA PyTorch
print('📦 Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.40.0', 'peft>=0.10.0', 'bitsandbytes>=0.43.0', 'datasets>=2.18.0', 'trl>=0.8.0', 'huggingface_hub', 'accelerate'], check=True)

# 4. Run Cloud Training Orchestrator
print('🚀 Launching Cloud Training Orchestrator...')
res = subprocess.run([sys.executable, 'scripts/cloud_train_orchestrator.py', '--platform', 'KAGGLE_T4', '--base-model', 'Qwen/Qwen2.5-7B-Instruct', '--epochs', '3'])
if res.returncode != 0:
    raise RuntimeError(f'❌ Training orchestrator failed with exit code {res.returncode}')
print('🎉 Training pipeline completed successfully!')
